# Week 1 Wed: Vectors, matrices, and linear algebra intuition for portfolios and ML

Estimated time: 4 hours

## Why this matters
Portfolio weights, factor exposures, and ML features are easiest to understand as vectors and matrices.

## Study blocks
- Block 1 (45 min): Learn scalars, vectors, and matrices through finance objects.
- Block 2 (60 min): Understand the dot product as portfolio return.
- Block 3 (60 min): Use matrices for scenarios, features, and multiple assets.
- Block 4 (45 min): Preview covariance and diversification.
- Block 5 (30 min): Code and interview drill.

## Concept notes
### Scalars, vectors, and matrices
A scalar is one number, a vector is an ordered list of numbers, and a matrix is a table of numbers.

Technical note: In quant work, vectors often store weights, returns, or factor exposures, while matrices store many observations across time or assets.

Finance example: A portfolio of three assets with weights [0.5, 0.3, 0.2] is naturally represented as a vector.

### Dot product as weighted sum
The dot product multiplies matching entries and adds them. In finance that gives a weighted return or exposure.

Technical note: If w is a weight vector and r is a return vector, portfolio return is w^T r.

Finance example: A portfolio return is just the sum of each asset's return times its portfolio weight.

### Matrices as data structures
A matrix helps store many assets over many dates or many features for many observations.

Technical note: An n by p matrix may represent n observations and p features for machine learning, or p assets over n dates for portfolio analytics.

Finance example: A return matrix with rows as days and columns as assets is the standard input to covariance and optimization work.


## Code Lab 1: Dot product and portfolio returns

In [ ]:
import numpy as np

weights = np.array([0.5, 0.3, 0.2])
asset_returns = np.array([0.02, -0.01, 0.03])
portfolio_return = weights @ asset_returns

print("Portfolio return:", round(portfolio_return, 4))


## Code Lab 2: Return matrix across multiple days

In [ ]:
import numpy as np
import pandas as pd

returns_matrix = np.array(
    [
        [0.01, -0.01, 0.02],
        [0.00, 0.02, -0.01],
        [0.03, -0.02, 0.01],
    ]
)
df = pd.DataFrame(returns_matrix, columns=["asset_a", "asset_b", "asset_c"])
df["portfolio_return"] = returns_matrix @ np.array([0.5, 0.3, 0.2])
print(df)


## Code Lab 3: Covariance preview

In [ ]:
print(df[["asset_a", "asset_b", "asset_c"]].cov())


## Practice recap
- What does a weight vector mean economically?
  Suggested answer: It tells you what fraction of the portfolio is allocated to each asset or strategy component.
- Why is the dot product useful in quant work?
  Suggested answer: Because many quantities are weighted sums: portfolio return, factor exposure, and expected payoff all fit that pattern.
- Why does covariance show up naturally after matrices?
  Suggested answer: Because once returns are organized in a matrix, covariance measures the pairwise co-movement across columns or assets.

## Interview drill
- Q: How would you explain a vector to a trader?
  A: It is just an organized list of related numbers, like a set of portfolio weights or factor exposures.
- Q: What is the finance interpretation of a dot product?
  A: It is usually a weighted sum, most commonly a portfolio return from weights and asset returns.

## 4+ Hour Completion Roadmap

Use this minimum structure to turn today's notebook into a serious study session:

- Phase 1 (60 min): Read concept notes and rewrite the core idea from memory.
- Phase 2 (70 min): Complete and extend all code labs with at least one variation.
- Phase 3 (65 min): Do formula retrieval, scenario analysis, and error-log updates.
- Phase 4 (65 min): Write a mini memo and practice interview responses aloud.

## Formula Rewrite Drill

In [ ]:
import pandas as pd

formula_drill = pd.DataFrame(
    [
        {"formula": "E[X] = sum_i p_i x_i", "from_memory": "", "term_explanations": ""},
        {"formula": "Var(X) = E[(X - E[X])^2]", "from_memory": "", "term_explanations": ""},
        {"formula": "W_t = W_0 * product(1 + r_t)", "from_memory": "", "term_explanations": ""},
        {"formula": "Topic formula for: Vectors, matrices, and linear algebra intuition for portfolios and ML", "from_memory": "", "term_explanations": ""},
    ]
)
print(formula_drill)


## Real Market Data Lab (Useful From Day 1)

This section uses a local CSV snapshot of real market prices so the notebook remains reproducible.

Dataset: `curriculum/datasets/real_market_prices.csv`
Symbols: SPY, QQQ, TLT, GLD

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

data_path = Path("curriculum/datasets/real_market_prices.csv")
market = pd.read_csv(data_path, parse_dates=["date"]).sort_values(["date", "symbol"])

print("Rows:", len(market))
print("Date range:", market["date"].min().date(), "to", market["date"].max().date())
print("Symbols:", sorted(market["symbol"].unique()))
print(market.head())

prices = market.pivot(index="date", columns="symbol", values="close").dropna()
returns = prices.pct_change().dropna()

summary = pd.DataFrame(
    {
        "ann_return": returns.mean() * 252,
        "ann_vol": returns.std() * (252 ** 0.5),
        "max_drawdown": ((1 + returns).cumprod().div((1 + returns).cumprod().cummax()) - 1).min(),
    }
).sort_index()
print("\nRisk/return summary:")
print(summary.round(4))

cum = (1 + returns).cumprod()
cum.plot(figsize=(10, 5), title="Cumulative growth (base 1.0)")
plt.ylabel("Growth of $1")
plt.tight_layout()
plt.show()


## Real-World Takeaway Prompt

Write 5-8 lines for today's topic (Vectors, matrices, and linear algebra intuition for portfolios and ML):

1. Which symbol looked most volatile and why that matters.
2. Which pair looked most diversifying.
3. One realistic portfolio/risk decision you could make from this table.

## Interview Question + Python Solution Drill

Use this format for interview preparation:

1. State the question clearly.
2. Explain your approach in plain language.
3. Write Python code on real market data.
4. Interpret one risk caveat in words.

**Suggested data-source ladder**
- Source 1: yfinance pull (fresh market data)
- Source 2: Robinhood-style CSV export (if available locally)
- Source 3: local snapshot `curriculum/datasets/real_market_prices.csv` (reproducible fallback)

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

prices = None
source_used = ""
USE_YFINANCE = False  # set True when you want a live market pull

try:
    import yfinance as yf
except ImportError:
    yf = None

if USE_YFINANCE and yf is not None:
    downloaded = yf.download(
        ["SPY", "QQQ", "TLT", "GLD"],
        period="2y",
        interval="1d",
        auto_adjust=True,
        progress=False,
    )
    if not downloaded.empty:
        if isinstance(downloaded.columns, pd.MultiIndex):
            if "Close" in downloaded.columns.get_level_values(0):
                prices = downloaded["Close"].copy()
            elif "Adj Close" in downloaded.columns.get_level_values(0):
                prices = downloaded["Adj Close"].copy()
        else:
            prices = downloaded.rename(columns=str.upper)
        source_used = "yfinance"

robinhood_export = Path("curriculum/datasets/robinhood_export.csv")
if prices is None and robinhood_export.exists():
    rh = pd.read_csv(robinhood_export, parse_dates=["date"])
    prices = rh.pivot(index="date", columns="symbol", values="close").sort_index()
    source_used = "robinhood_export_csv"

if prices is None:
    local = pd.read_csv(
        Path("curriculum/datasets/real_market_prices.csv"),
        parse_dates=["date"],
    )
    prices = local.pivot(index="date", columns="symbol", values="close").sort_index()
    source_used = "local_snapshot_csv"

prices = prices.dropna(how="all").ffill().dropna()
returns = prices.pct_change().dropna()
log_returns = np.log(prices / prices.shift(1)).dropna()

annualized_return = returns.mean() * 252
annualized_vol = returns.std() * np.sqrt(252)
sharpe_proxy = (annualized_return - 0.03) / annualized_vol

print("Source used:", source_used)
print("\nAnnualized return:")
print(annualized_return.round(4))
print("\nAnnualized volatility:")
print(annualized_vol.round(4))
print("\nSharpe proxy (rf=3%):")
print(sharpe_proxy.round(3))
print("\nRecent log returns:")
print(log_returns.tail().round(4))


## Scenario Analysis Drill

In [ ]:
import pandas as pd

scenarios = pd.DataFrame(
    [
        {"scenario": "base_case", "assumption": "", "expected_effect": ""},
        {"scenario": "bull_case", "assumption": "", "expected_effect": ""},
        {"scenario": "stress_case", "assumption": "", "expected_effect": ""},
    ]
)
print("Topic:", 'Vectors, matrices, and linear algebra intuition for portfolios and ML')
print(scenarios)


## Closed-Book Retrieval and Error Log

In [ ]:
import pandas as pd

retrieval_scorecard = pd.DataFrame(
    [
        {"prompt": "Explain today's concept in 3 lines", "score_0_to_5": None, "notes": ""},
        {"prompt": "Write one key formula without notes", "score_0_to_5": None, "notes": ""},
        {"prompt": "Give one realistic failure mode", "score_0_to_5": None, "notes": ""},
        {"prompt": "Connect to one trading/risk decision", "score_0_to_5": None, "notes": ""},
    ]
)

error_log = pd.DataFrame(
    [
        {
            "concept": "",
            "mistake": "",
            "correction": "",
            "next_review_date": "",
        }
    ]
)
print("Retrieval scorecard template:")
print(retrieval_scorecard)
print("\nError log template:")
print(error_log)


## Final 30-Minute Deliverable

Write a short memo (150-250 words) with this structure:

1. Core idea of Vectors, matrices, and linear algebra intuition for portfolios and ML in plain language.
2. One technical detail or formula and why it matters.
3. One practical quant use case.
4. One limitation or failure mode and how you would detect it.